In [ ]:
import h5py
import pandas as pd
import numpy as np
import os
from pathlib import Path
from matplotlib import pyplot as plt
import scipy.stats
from scipy.stats import wilcoxon
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle, Polygon
from matplotlib.collections import PolyCollection
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from functools import partial
import warnings
import vbn_utils
import decoding_utils as du
from analysis_utils import exponential_convolve
import ccf_utils
from vbn_utils import cumulative_hist, formatFigure, mean_sem_plot, make_iterable, get_unit_ids, bootstrap_ci
%matplotlib inline

## Data loading

In [ ]:
#Paths to all of the useful supplemental tables and tensors
active_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor.hdf5"
passive_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor_passive.hdf5"
#opto_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbn_opto_tensor_unit_grouped.hdf5"

stim_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_stim_table_no_filter.csv"
unit_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_units_with_responsiveness.csv"

sessions_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_sessions_table.csv" #"/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/vbn_s3_cache/visual-behavior-neuropixels-0.5.0/project_metadata/ecephys_sessions.csv"

In [ ]:
units = pd.read_csv(unit_table_file)
units['cortical_layer'] = units['cortical_layer'].replace('3-Feb','2/3') # necessary since 2/3 sometimes gets incorrectly reformatted as a date

stim_table = pd.read_csv(stim_table_file)
stim_table = stim_table.drop(columns='Unnamed: 0')

active_tensor = h5py.File(active_tensor_file)

sessions_table = pd.read_csv(sessions_table_file)

In [ ]:
new_clusters = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/cluster_labels_122024.csv")
units = units.merge(new_clusters[['unit_id', 'cluster_labels_new_weak_cluster_12', 'used_in_clustering']], on='unit_id', how='left')
units.rename(columns={'cluster_labels_new_weak_cluster_12': 'cluster_labels_new', 'used_in_clustering': 'used_in_new_clustering'}, inplace=True)

In [ ]:
structure_tree = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/ccf_structure_tree_2017.csv")

In [ ]:
stim_table['is_shared'] = stim_table['image_name'].isin(['im083_r', 'im111_r'])

## Time from last lick

### Cluster-wise responses as function of time from last lick (not running matched)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

fig_agg, ax_agg = plt.subplots()

base_sub = False
session_list = list(active_tensor.keys())
time = np.arange(-750, 750)

cluster_colors = {
    1: 'navy',  # Blue
    2: 'mediumblue',  # Lighter blue
    3: 'royalblue',  # Even lighter blue
    4: 'cornflowerblue',  # Lighter blue
    5: 'slateblue',  # Lightest blue
    6: 'red',  # Red
    7: 'coral',  # Lighter red
    8: 'gold'   # Lightest red
}


for cluster in np.arange(6,8):
# flash_data = {cluster:[] for cluster in np.arange(15)}
    unit_filter = du.get_units_in_cluster(units, cluster, clustering='new') & du.apply_unit_quality_filter(units) & du.getUnitsInRegion(units, 'SCMRN')
    unit_ids = units.loc[unit_filter]['unit_id'].values #[1076896692, 1068232021, 1068231696,]#
    num_sessions = units.loc[unit_filter]['ecephys_session_id'].unique().size


    fig, ax = plt.subplots()
    fig.set_size_inches(10,10)
    fig.suptitle(f'cluster {cluster}, units {len(unit_ids)}, sessions {num_sessions}')
    stim_filter_cond = ['engaged', '~is_change', '~omitted', '~previous_omitted', 'flashes_since_change>5']
    #stim_filter_cond2 = ['engaged', '~is_change', '~omitted', '~previous_omitted', 'flashes_since_change>5', 'flashes_since_last_lick>1', 'lickbout_for_flash_during_response_window', "image_name=='im036_r'"]

    cluster_means = []
    for flashes_since_lick in range(11):
        if flashes_since_lick==0:
            stim_filter = ['engaged', 'is_change', 'hit']
            label = 'change lick'
            alpha = 1
            color = 'g'
            ls = 'solid'
        elif flashes_since_lick==1:
            continue
        elif flashes_since_lick<10:
            filter_add_on = [f'flashes_since_last_lick=={flashes_since_lick}', '~lickbout_for_flash_during_response_window']
            stim_filter =  stim_filter_cond + filter_add_on
            label = f'{flashes_since_lick}'
            alpha = flashes_since_lick/9
            color = 'k'
            ls = 'solid'
        elif flashes_since_lick==10: 
            filter_add_on = ['lickbout_for_flash_during_response_window']
            stim_filter =  stim_filter_cond + filter_add_on
            label = 'non change lick'
            color = 'g'
            alpha = 1
            ls = 'dotted'

        unit_data, unitIds = vbn_utils.unit_averaged_psth(active_tensor_file, 
                                    stim_table, 
                                    session_list, 
                                    unit_ids, 
                                    *stim_filter, 
                                    baseline_length=750, 
                                    resp_window_length=750)
        concat = np.concatenate(unit_data)
        concat = np.array([exponential_convolve(c, 5, symmetrical=True) for c in concat])
        if concat.size==0:
            continue
        
        if base_sub:
            concat = concat - concat[:, :50].mean(axis=1)[:, None]
        # concat = np.array([np.nanmean(u, axis=0) for u in unit_data])
        # flash_data[cluster].append(concat)
        #vbn_utils.mean_sem_plot(concat*1000, ax, time,)# color=colors[istim])
        ax.plot(time, np.nanmean(concat, axis=0), label=label, color=color, alpha=alpha, ls=ls,)
        ax.set_xlabel('Time from flash (ms)')
        cluster_means.append(np.nanmean(concat[:, 650:750], axis=1))
    
    cluster_means = np.array(cluster_means)
    cluster_means = np.roll(cluster_means, shift=-1, axis=0)
    cluster_means = cluster_means - cluster_means[0][None,:]
    cluster_mean = np.nanmean(cluster_means, axis=1)
    cluster_sem = np.nanstd(cluster_means, axis=1)/cluster_means.shape[1]**0.5
    ax_agg.plot(np.arange(2,12), cluster_mean, 'o', color=cluster_colors[cluster], label=cluster)
    ax_agg.errorbar(np.arange(2,12), cluster_mean, yerr=cluster_sem, color=cluster_colors[cluster])
    ax_agg.set_xticks(np.arange(2,12))
    ax_agg.set_xticklabels(list(np.arange(2,10)) + ['false alarm', 'hit'])
    ax.set_xlim(-400, 350)
    ax.legend()
    ax_agg.legend()

### Cluster-wise responses as function of time from last lick (running matched)

In [ ]:
running_df = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/all_running_dfs.csv")

In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 15

In [ ]:
import warnings
warnings.filterwarnings("ignore")

plt.rcParams['font.size'] = 15

fig_agg, ax_agg = plt.subplots()

cluster_colors = {
    1: 'navy',  # Blue
    2: 'mediumblue',  # Lighter blue
    3: 'royalblue',  # Even lighter blue
    4: 'cornflowerblue',  # Lighter blue
    5: 'slateblue',  # Lightest blue
    6: 'red',  # Red
    7: 'coral',  # Lighter red
    8: 'gold'   # Lightest red
}

cluster_names = {
	1: 'On Transient',
	2: 'On Sustained',
	3: 'Off Transient',
	4: 'Off Sustained',
	5: 'Image suppressed',
	6: 'Lick anticipation',
	7: 'Licking',
	8: 'Running'
}

time = np.arange(-750,750)

all_cluster_means = {}
for cluster in range(6,8):
	fig, ax = plt.subplots()
	fig.suptitle(cluster_names[cluster])
	unit_filter = du.get_units_in_cluster(units, cluster, clustering='new') & du.apply_unit_quality_filter(units) & \
			du.getUnitsInRegion(units, 'SCMRN')
			
	unit_ids = units.loc[unit_filter]['unit_id'].values
	if len(unit_ids)==0:
		continue
	session_list = units.loc[unit_filter]['ecephys_session_id'].unique()

	max_flashes = 7

	stim_filter_base = ['engaged', '~is_change', '~omitted', '~previous_omitted', '~grace_period_after_hit']# 'flashes_since_change>5']
	cond_filters = [
			['engaged', 'is_change', 'hit'],
			['lickbout_for_flash_during_response_window'] + stim_filter_base,
			['engaged', 'is_change', 'miss']
			] + [[f'flashes_since_last_lick=={flashes_since_lick}', '~lickbout_for_flash_during_response_window'] + 
					stim_filter_base for flashes_since_lick in range(2, max_flashes+1)]

	unit_data, stim_indices, unitIds, returned_session_ids = vbn_utils.unit_averaged_psth_col_matched(active_tensor_file, stim_table, session_list, unit_ids, cond_filters, 'baseline_running', 
							baseline_length=750, resp_window_length=750, num_iterations=1)
	
	passive_unit_data, passive_stim_indices, passive_unitIds, passive_returned_session_ids = vbn_utils.unit_averaged_psth_col_matched(passive_tensor_file, stim_table, session_list, unit_ids, cond_filters, 'baseline_running', 
							baseline_length=750, resp_window_length=750, num_iterations=1)
	
	unit_data = [u for u in unit_data if u.size>0]
	unit_data = np.concatenate(unit_data, axis=1)
	passive_unit_data = [u for u in passive_unit_data if u.size>0]
	passive_unit_data = np.concatenate(passive_unit_data, axis=1)
	cluster_means = []
	for icond, cond in enumerate(unit_data):
		if icond==0:
			label = 'change lick'
			alpha = 1
			color = 'g'
			ls = 'solid'
		
		elif icond==1: 
			label = 'non change lick'
			color = 'g'
			alpha = 1
			ls = 'dotted'
		
		elif icond== 2:
			label = 'miss'
			alpha = 1
			color = 'goldenrod'
			ls = 'solid'

		elif icond>2:
			label = f'{icond-1} flashes since lick'
			alpha = (icond-1)/max_flashes
			color = 'k'
			ls = 'solid'
		
		
		
		cond = np.array([exponential_convolve(c, 7, symmetrical=True) for c in cond])
		ax.plot(time, np.nanmean(cond, axis=0)*1000, color=color, alpha=alpha, ls=ls,)# label=icond)
		cluster_means.append(np.nanmean(cond[:, 550:750]*1000, axis=1))
    
	cluster_means = np.array(cluster_means)
	cluster_means = np.roll(cluster_means, shift=-3, axis=0)
	all_cluster_means[cluster] = cluster_means
	cluster_means = cluster_means - cluster_means[0][None,:]
	cluster_mean = np.nanmean(cluster_means, axis=1)
	cluster_sem = np.nanstd(cluster_means, axis=1)/cluster_means.shape[1]**0.5
	# ax_agg.plot(np.arange(2,max_flashes+3), cluster_mean, 'o', color=cluster_colors[cluster],)# label=cluster)
	# ax_agg.errorbar(np.arange(2,max_flashes+3), cluster_mean, yerr=cluster_sem, color=cluster_colors[cluster])

	ax_agg.plot(np.arange(2,max_flashes+1), cluster_mean[:-3], 'o', color=cluster_colors[cluster], label=cluster_names[cluster])
	ax_agg.errorbar(np.arange(2,max_flashes+1), cluster_mean[:-3], yerr=cluster_sem[:-3], color=cluster_colors[cluster])
	ax_agg.plot(np.arange(max_flashes+1,max_flashes+4), cluster_mean[-3:], 'o', color=cluster_colors[cluster],)# label=cluster)
	ax_agg.errorbar(np.arange(max_flashes+1,max_flashes+4), cluster_mean[-3:], yerr=cluster_sem[-3:], color=cluster_colors[cluster])
	
	# ax.set_ylim(0.01, 0.04)
	ax.set_xlim(-300, 500)
	ax.set_xlabel('Time from image onset (ms)')
	ax.set_ylabel('Firing rate (Hz)')

	passive_hit = np.array([exponential_convolve(c, 7, symmetrical=True) for c in passive_unit_data[0]])
	ax.plot(time, np.nanmean(passive_hit, axis=0)*1000, color='g', alpha=0.5, ls='-.', label='passive hit')
	ax.legend(['hit', 'false alarm', 'miss'] + list(np.arange(2,max_flashes+1)) + ['passive hit',], frameon=False, loc='upper left', bbox_to_anchor=(0, 1.05))


	vbn_utils.formatFigure(fig, ax)

ax_agg.set_xticks(np.arange(2,max_flashes+4))
ax_agg.set_xticklabels(list(np.arange(2,max_flashes+1)) + ['hit', 'false alarm', 'miss'])
for il, label in enumerate(ax_agg.get_xticklabels()):
	if il > max_flashes-2:
		label.set_rotation(30)

ax_agg.legend(frameon=False)#labels=['Lick anticipation cluster', 'Licking cluster'])
ax_agg.set_xlabel('Image presentations since last lick')
ax_agg.set_ylabel('Baseline firing rate (Hz)')
vbn_utils.formatFigure(fig_agg, ax_agg)

In [ ]:
from nwb_session_utils import resample_df_to_times

session_cond_running = []
for session_id, stim_inds in zip(returned_session_ids, stim_indices):

    if len(stim_inds[0])==0:
        continue
    
    session_stims = stim_table[stim_table['session_id']==session_id].reset_index()
    session_running = running_df[running_df['session_id']==session_id]

    cond_running = []
    for cond_inds in stim_inds[0]:
        start_times = session_stims.loc[cond_inds]['start_time'].values
        
        stim_running_traces = []
        for start in start_times:
            time_vector = np.linspace(start*1000-1000, start*1000+1200, 2200)
            stim_running, _ = resample_df_to_times(session_running, 'timestamps', 'speed', time_vector)
            stim_running_traces.append(stim_running[:2000])
        
        cond_running.append(np.nanmean(stim_running_traces,axis=0))
    
    session_cond_running.append(np.array(cond_running))

In [ ]:
session_cond_running = np.array([s for s in session_cond_running if s.size>8])

In [ ]:
plt.figure()
for cond in range(8):
    plt.plot(np.nanmean(session_cond_running[:, cond], axis=0))

In [ ]:
from scipy.stats import wilcoxon
for ic, c in all_cluster_means.items():
    
    plt.figure(ic)
    plt.plot(c[0], c[-1], 'k.')
    plt.plot(np.nanmean(c[0]), np.nanmean(c[-1]), 'ro')
    plt.plot([0,0.1], [0,0.1], 'k--')

    p_res = wilcoxon(c[0], c[-1], nan_policy='omit')
    plt.title(f'{ic} {p_res} \n n={np.sum(~np.isnan(c[0]))}')

## Analysis of behavior since last lick

In [ ]:
stims = stim_table[stim_table['no_abnorm']]

In [ ]:
fig, ax = plt.subplots()
table = stims[stims['engaged']]
ax.plot(table.pivot_table(index='flashes_since_last_lick', values='is_change', aggfunc='mean'), 'ko-',)
ax.set_xlim(0, 15.5)
ax.set_xlabel('Image presentations since last lick')
ax.set_ylabel('change probability')
ax.set_ylim(-0.01,0.35)

ax2 = ax.twinx()
table = stims[(~stims['is_change'])&(~stims['omitted'])&(stims['engaged'])&(~stims['previous_omitted'])]
ax2.plot(table.pivot_table(index='flashes_since_last_lick', 
            values='lickbout_for_flash_during_response_window', aggfunc='mean'), 'ro-',)
ax2.set_xlim(0, 15.5)
# ax2.set_xlabel('Image presentations since last lick')
ax2.set_ylabel('false alarm rate', color='r')
ax2.set_ylim(-0.01,0.35)
ax2.tick_params(axis='y', colors='r')

[a.spines['top'].set_visible(False) for a in [ax, ax2]]

In [ ]:
stims['rt_zscore'] = stims.groupby('session_id')['reaction_time'].transform(lambda x: (x-np.nanmean(x))/np.nanstd(x))

fig, ax = plt.subplots()
hit_table = stims[stims['is_change']&stims['engaged']&(stims['flashes_since_last_lick']>1)&(stims['flashes_since_last_lick']<16)]
vals = hit_table.pivot_table(index='flashes_since_last_lick', values='rt_zscore', aggfunc=('mean', 'sem'))
vals.plot(y='mean', yerr='sem', kind='line', marker='o', color='k', ax=ax, label='hits')


fa_table = stims[(stims['engaged'])&(~stims['is_change'])&(~stims['omitted'])&(~stims['previous_omitted'])&(~stims['grace_period_after_hit']) &\
    (stims['flashes_since_change']>1)&(stims['flashes_since_last_lick']>1)&(stims['flashes_since_last_lick']<16)]
vals = fa_table.pivot_table(index='flashes_since_last_lick', values='rt_zscore', aggfunc=('mean', 'sem'))
vals.plot(y='mean', yerr='sem', kind='line', marker='o', color='r', ax=ax, label='false alarms')
ax.set_xlim(0,15.5)
ax.set_ylim(-0.55,0.5)
ax.set_xlabel('Image presentations since last lick')
ax.set_ylabel('z-scored response time')

plt.legend()
vbn_utils.formatFigure(fig, ax)

fig, ax = plt.subplots()
hit_table = stims[stims['is_change']&stims['engaged']&(stims['flashes_since_last_lick']>1)&(stims['flashes_since_last_lick']<16)]
vals = hit_table.pivot_table(index='flashes_since_last_lick', values='reaction_time', aggfunc=('mean', 'sem'))
vals.plot(y='mean', yerr='sem', kind='line', marker='o', color='k', ax=ax, label='hits')


fa_table = stims[(stims['engaged'])&(~stims['is_change'])&(~stims['omitted'])&(~stims['previous_omitted'])&(~stims['grace_period_after_hit']) &\
    (stims['flashes_since_change']>1)&(stims['flashes_since_last_lick']>1)&(stims['flashes_since_last_lick']<16)]
vals = fa_table.pivot_table(index='flashes_since_last_lick', values='reaction_time', aggfunc=('mean', 'sem'))
vals.plot(y='mean', yerr='sem', kind='line', marker='o', color='r', ax=ax, label='false alarms')
ax.set_xlim(0,15.5)
# ax.set_ylim(-0.55,0.5)
ax.set_xlabel('Image presentations since last lick')
ax.set_ylabel('Response time (s)')
ax.set_ylim(0.35, 0.5)
plt.legend()
vbn_utils.formatFigure(fig, ax)


## GLM Kernel-subtracted PETH

In [ ]:
import nwb_session_utils as nwb
import warnings
warnings.filterwarnings("ignore")

unit_filter = du.get_units_in_cluster(units, *np.arange(1,9), clustering='new') & du.apply_unit_quality_filter(units)
		
unit_ids = units.loc[unit_filter]['unit_id'].values

session_list = units.loc[unit_filter]['ecephys_session_id'].unique()

stim_filter_base = ['engaged', '~is_change', '~omitted', '~previous_omitted', 'flashes_since_change>5']
cond_filters = [
		['engaged', 'is_change', 'hit'],
		['lickbout_for_flash_during_response_window'] + stim_filter_base,
		] + [[f'flashes_since_last_lick=={flashes_since_lick}', '~lickbout_for_flash_during_response_window'] + 
				stim_filter_base for flashes_since_lick in range(2, 10)]

raw_firing_rates, kernel_subtracted_firing_rates, running_traces, unitIds = nwb.unit_kernel_subtracted_peth(session_list, unit_ids, cond_filters)

In [ ]:
kernel_subtracted_summary = {}
for raws, subs, runs, uids in zip(raw_firing_rates, kernel_subtracted_firing_rates, running_traces, unitIds):
    for raw, sub, uid in zip(raws, subs, uids):
        kernel_subtracted_summary[uid] = {'raw': raw, 'subtracted': sub, 'running': runs}

In [ ]:
import warnings
warnings.filterwarnings("ignore")

fig_agg, ax_agg = plt.subplots()

cluster_colors = {
    1: 'navy',  # Blue
    2: 'mediumblue',  # Lighter blue
    3: 'royalblue',  # Even lighter blue
    4: 'cornflowerblue',  # Lighter blue
    5: 'slateblue',  # Lightest blue
    6: 'red',  # Red
    7: 'tomato',  # Lighter red
    8: 'coral'   # Lightest red
}

for cluster in range(1,9):
	fig, ax = plt.subplots(1,4)
	fig.set_size_inches(10,6)
	fig.suptitle(cluster)
	unit_filter = du.get_units_in_cluster(units, cluster, clustering='new') & du.apply_unit_quality_filter(units) & \
			du.getUnitsInRegion(units, 'SCMRN')
			
	unit_ids = units.loc[unit_filter]['unit_id'].values
	if len(unit_ids)==0:
		continue
	
	unit_data_raw = np.array([kernel_subtracted_summary[uid]['raw'] for uid in unit_ids])[:,:,:80]
	unit_data_sub = np.array([kernel_subtracted_summary[uid]['subtracted'] for uid in unit_ids])[:,:,:80]
	unit_running = np.array([kernel_subtracted_summary[uid]['running'] for uid in unit_ids])[:,:,:80]

	session_list = units.loc[unit_filter]['ecephys_session_id'].unique()

	stim_filter_base = ['engaged', '~is_change', '~omitted', '~previous_omitted', 'flashes_since_change>5']
	cond_filters = [
			['engaged', 'is_change', 'hit'],
			['lickbout_for_flash_during_response_window'] + stim_filter_base,
			] + [[f'flashes_since_last_lick=={flashes_since_lick}', '~lickbout_for_flash_during_response_window'] + 
					stim_filter_base for flashes_since_lick in range(2, 10)]

	
	cluster_means = []
	for icond in range(unit_data_raw.shape[1]):
		if icond==0:
			label = 'change lick'
			alpha = 1
			color = 'g'
			ls = 'solid'

		elif icond>1:
			label = f'{icond}'
			alpha = icond/9
			color = 'k'
			ls = 'solid'
		elif icond==1: 
			label = 'non change lick'
			color = 'g'
			alpha = 1
			ls = 'dotted'

		#cond = np.array([exponential_convolve(c, 3, symmetrical=True) for c in cond])
		ax[0].plot(np.nanmean(unit_data_raw[:, icond], axis=0), color=color, alpha=alpha, ls=ls, label=icond)
		ax[1].plot(np.nanmean(unit_data_sub[:, icond], axis=0), color=color, alpha=alpha, ls=ls, label=icond)
		ax[2].plot(np.nanmean(unit_running[:, icond], axis=0), color=color, alpha=alpha, ls=ls, label=icond)
		cluster_means.append(np.nanmean(unit_data_sub[:, icond, 34:42], axis=1))
    
	cluster_means = np.array(cluster_means)
	cluster_means = np.roll(cluster_means, shift=-2, axis=0)
	cluster_means = cluster_means - cluster_means[0][None,:]
	cluster_mean = np.nanmean(cluster_means, axis=1)
	cluster_sem = np.nanstd(cluster_means, axis=1)/cluster_means.shape[1]**0.5
	ax_agg.plot(np.arange(2,12), cluster_mean, 'o', color=cluster_colors[cluster], label=cluster)
	ax_agg.errorbar(np.arange(2,12), cluster_mean, yerr=cluster_sem, color=cluster_colors[cluster])
	
	ax[0].legend()
	# ax.set_ylim(0.01, 0.04)
	[a.set_xlim(20, 50) for a in ax[:3]]

	ax[3].plot(np.nanmean(unit_running[:, 0, 34:42], axis=1), np.nanmean(unit_running[:, -1, 34:42], axis=1), 'ko')
	ax[3].plot(np.nanmean(unit_running[:, 1, 34:42], axis=1), np.nanmean(unit_running[:, -1, 34:42], axis=1), 'ro')
	# ax[3].plot([0,100], [0,100], 'k--')
	ax[3].plot([-1,1], [-1,1], 'k--')

ax_agg.set_xticks(np.arange(2,12))
ax_agg.set_xticklabels(list(np.arange(2,10)) + ['hit', 'false alarm'])
ax_agg.legend()

## Responses by reaction time quintile

### Not running matched

In [ ]:
import warnings
warnings.filterwarnings("ignore")

fig_rt, ax_rt = plt.subplots()
for q in range(5):

    rts = stim_table[stim_table['rt_quintiles']==q]['reaction_time'].values
    ax_rt.boxplot(rts, notch=True, positions=[q,], showfliers=False)

fig_agg, ax_agg = plt.subplots()

base_sub = False
session_list = list(active_tensor.keys())
time = np.arange(-750, 750)

cluster_colors = {
    1: 'navy',  # Blue
    2: 'mediumblue',  # Lighter blue
    3: 'royalblue',  # Even lighter blue
    4: 'cornflowerblue',  # Lighter blue
    5: 'slateblue',  # Lightest blue
    6: 'red',  # Red
    7: 'coral',  # Lighter red
    8: 'gold'   # Lightest red
}

cluster_names = {
	1: 'On Transient',
	2: 'On Sustained',
	3: 'Off Transient',
	4: 'Off Sustained',
	5: 'Image suppressed',
	6: 'Lick anticipation',
	7: 'Licking',
	8: 'Running'
}

for cluster in np.arange(6,8):
# flash_data = {cluster:[] for cluster in np.arange(15)}
    unit_filter = du.get_units_in_cluster(units, cluster, clustering='new') & du.apply_unit_quality_filter(units) & \
        du.getUnitsInRegion(units, 'SCMRN')
    unit_ids = units.loc[unit_filter]['unit_id'].values #[1076896692, 1068232021, 1068231696,]#
    if len(unit_ids)<30:
        continue
    num_sessions = units.loc[unit_filter]['ecephys_session_id'].unique().size


    # fig, axes = plt.subplots(1,2)
    # ax = axes[0]
    fig, ax = plt.subplots()
    fig.set_size_inches(12,6)
    fig.suptitle(f'cluster {cluster}, units {len(unit_ids)}, sessions {num_sessions}')
    stim_filter_cond = ['engaged', 'is_change', '~omitted', '~previous_omitted',]# 'flashes_since_change>5']
    #stim_filter_cond2 = ['engaged', '~is_change', '~omitted', '~previous_omitted', 'flashes_since_change>5', 'flashes_since_last_lick>1', 'lickbout_for_flash_during_response_window', "image_name=='im036_r'"]

    cluster_means = []
    for reaction_time_quintile in range(8):
        if reaction_time_quintile==5:
            stim_filter = ['engaged', 'is_change', 'hit']
            label = 'hit'
            alpha = 0
            color = 'g'
            ls = 'solid'
        
        elif reaction_time_quintile==6:
            filter_add_on = ['engaged', '~is_change', '~omitted', '~previous_omitted',
                            '~lickbout_for_flash_during_response_window', 'flashes_since_last_lick>1', 'flashes_since_change>5']
            stim_filter =  filter_add_on
            label = 'correct reject'
            color = 'g'
            alpha = 1
            ls = 'dotted'

        elif reaction_time_quintile==7:
            stim_filter = ['engaged', 'is_change', 'miss']
            label = 'miss'
            alpha = 0.5
            color = 'g'
            ls = 'solid'

        else:
            filter_add_on = [f'rt_quintiles=={reaction_time_quintile}', 'lickbout_for_flash_during_response_window', 
                            'reaction_time>0.1']
            stim_filter =  stim_filter_cond + filter_add_on
            label = f'{reaction_time_quintile}'
            alpha = 1.2-(reaction_time_quintile+1)/5
            color = 'k'
            ls = 'solid'
        

        unit_data, unitIds = vbn_utils.unit_averaged_psth(active_tensor_file, 
                                    stim_table, 
                                    session_list, 
                                    unit_ids, 
                                    *stim_filter, 
                                    baseline_length=750, 
                                    resp_window_length=750)
        concat = np.concatenate(unit_data)
        concat = np.array([exponential_convolve(c, 7, symmetrical=True) for c in concat])
        if concat.size==0:
            continue
        
        if base_sub:
            concat = concat - concat[:, :50].mean(axis=1)[:, None]
        # concat = np.array([np.nanmean(u, axis=0) for u in unit_data])
        # flash_data[cluster].append(concat)
        #vbn_utils.mean_sem_plot(concat*1000, ax, time,)# color=colors[istim])
        ax.plot(time, np.nanmean(concat, axis=0), label=label, color=color, alpha=alpha, ls=ls,)
        # axes[1].plot(time[600:1200], np.nanmean(np.nancumsum(concat[:, 600:1200], axis=1), axis=0), label=label, color=color, alpha=alpha, ls=ls,)
        ax.set_xlabel('Time from flash (ms)')
        cluster_means.append(np.nanmean(concat[:, 650:750], axis=1))
    
    cluster_means = np.array(cluster_means)
    # cluster_means = np.roll(cluster_means, shift=-3, axis=0)
    cluster_means = cluster_means - cluster_means[0][None,:]
    cluster_mean = np.nanmean(cluster_means, axis=1)
    cluster_sem = np.nanstd(cluster_means, axis=1)/cluster_means.shape[1]**0.5
    ax_agg.plot(np.arange(8), cluster_mean, 'o', color=cluster_colors[cluster], label=cluster)
    ax_agg.errorbar(np.arange(8), cluster_mean, yerr=cluster_sem, color=cluster_colors[cluster])
    ax_agg.set_xticks(np.arange(8))
    ax_agg.set_xticklabels(list(np.arange(5)) + ['hit', 'correct reject', 'miss'])
    ax.set_xlim(-400, 750)
    ax.legend()
    ax_agg.legend()

### Running matched

In [ ]:
import warnings
warnings.filterwarnings("ignore")

fig_rt, ax_rt = plt.subplots()
for q in range(5):

    rts = stim_table[stim_table['rt_quintiles']==q]['reaction_time'].values
    ax_rt.boxplot(rts, notch=True, positions=[q,], showfliers=False)
ax_rt.set_xticks(np.arange(5))
ax_rt.set_xticklabels(np.arange(1,6))
ax_rt.set_ylabel('Response time (s)')
ax_rt.set_xlabel('Response time quintile')
vbn_utils.formatFigure(fig_rt, ax_rt)

fig_agg, ax_agg = plt.subplots()

cluster_colors = {
    1: 'navy',  # Blue
    2: 'mediumblue',  # Lighter blue
    3: 'royalblue',  # Even lighter blue
    4: 'cornflowerblue',  # Lighter blue
    5: 'slateblue',  # Lightest blue
    6: 'red',  # Red
    7: 'coral',  # Lighter red
    8: 'gold'   # Lightest red
}

cluster_names = {
	1: 'On Transient',
	2: 'On Sustained',
	3: 'Off Transient',
	4: 'Off Sustained',
	5: 'Image suppressed',
	6: 'Lick anticipation',
	7: 'Licking',
	8: 'Running'
}

time = np.arange(-750, 750)
for cluster in range(6,8):
	fig, ax = plt.subplots()
	fig.suptitle(cluster_names[cluster])
	# fig.set_size_inches(12,6)
	unit_filter = du.get_units_in_cluster(units, cluster, clustering='new') & du.apply_unit_quality_filter(units) & \
			du.getUnitsInRegion(units, 'SCMRN')
			
	unit_ids = units.loc[unit_filter]['unit_id'].values
	if len(unit_ids)==0:
		continue
	session_list = units.loc[unit_filter]['ecephys_session_id'].unique()

	max_flashes = 7

	stim_filter_base = ['engaged', 'is_change', '~omitted', '~previous_omitted',]# 'flashes_since_change>5']
	cond_filters = [
			['engaged', 'is_change', 'hit'],
			['engaged', '~is_change', '~omitted', '~previous_omitted',
                '~lickbout_for_flash_during_response_window', 'flashes_since_last_lick>1', 'flashes_since_change>5'],
			] + [[f'rt_quintiles=={rt_quintile}', 'lickbout_for_flash_during_response_window'] + 
					stim_filter_base for rt_quintile in range(5)]

	unit_data, stim_indices, unitIds, returned_session_ids = vbn_utils.unit_averaged_psth_col_matched(active_tensor_file, 
							stim_table, session_list, unit_ids, cond_filters, 'baseline_running', 
							baseline_length=750, resp_window_length=750, num_iterations=5)
	
	unit_data = [u for u in unit_data if u.size>0]
	unit_data = np.concatenate(unit_data, axis=1)
	cluster_means = []
	for icond, cond in enumerate(unit_data):
		if icond==0:
			label = 'hit'
			alpha = 0
			color = 'g'
			ls = 'solid'
		
		elif icond==1: 
			label = 'correct reject'
			color = 'g'
			alpha = 1
			ls = 'dotted'
		
		elif icond>=2:
			label = f'{icond-2}'
			alpha = 1.2-(icond-1)/5
			color = 'k'
			ls = 'solid'
		
		cond = np.array([exponential_convolve(c, 7, symmetrical=True) for c in cond])
		if icond>0:
			ax.plot(time, np.nanmean(cond, axis=0)*1000, color=color, alpha=alpha, ls=ls, label=label)
		cluster_means.append(np.nanmean(cond[:, 550:750]*1000, axis=1))
    
	ax.set_xlabel('Time from flash (ms)')

	cluster_means = np.array(cluster_means)
	cluster_means = np.roll(cluster_means, shift=-2, axis=0)
	cluster_means = cluster_means - cluster_means[0][None,:]

	cluster_means_norm = cluster_means/np.nanmax(np.abs(cluster_means), axis=0)[None, :]

	cluster_mean = np.nanmean(cluster_means, axis=1)
	cluster_sem = np.nanstd(cluster_means, axis=1)/cluster_means.shape[1]**0.5
	# ax_agg.plot(np.arange(7), cluster_mean, 'o', color=cluster_colors[cluster], label=cluster)
	# ax_agg.errorbar(np.arange(7), cluster_mean, yerr=cluster_sem, color=cluster_colors[cluster])
	ax_agg.plot(np.arange(5), cluster_mean[:-2], 'o', color=cluster_colors[cluster], label=cluster_names[cluster])
	ax_agg.errorbar(np.arange(5), cluster_mean[:-2], yerr=cluster_sem[:-2], color=cluster_colors[cluster])
	ax_agg.plot(np.arange(5,7), cluster_mean[-2:], 'o', color=cluster_colors[cluster],)# label=cluster)
	ax_agg.errorbar(np.arange(5,7), cluster_mean[-2:], yerr=cluster_sem[-2:], color=cluster_colors[cluster])
	
	ax.legend(['correct reject',] + list(np.arange(5)+1), frameon=False, loc='upper left', bbox_to_anchor=(0, 1.05))
	# ax.set_ylim(0.01, 0.04)
	ax.set_xlim(-400, 750)
	ax.set_xlabel('Time from image onset (ms)')
	ax.set_ylabel('Firing rate (Hz)')
	vbn_utils.formatFigure(fig, ax)


ax_agg.set_xticks(np.arange(7))
ax_agg.set_xticklabels(list(np.arange(5)+1) + ['hit', 'cr'])
ax_agg.legend(frameon=False, loc='upper left', bbox_to_anchor=(0, 0.2))#labels=['Lick anticipation cluster', 'Licking cluster'], frameon=False, loc='upper left', bbox_to_anchor=(0, 1.05)))
ax_agg.set_xlabel('Response time quintile')
ax_agg.set_ylabel('Baseline firing rate (Hz)')
vbn_utils.formatFigure(fig_agg, ax_agg)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# fig_agg, ax_agg = plt.subplots()

base_sub = False
session_list = list(active_tensor.keys())
time = np.arange(-750, 750)

cluster_colors = {
    1: 'navy',  # Blue
    2: 'mediumblue',  # Lighter blue
    3: 'royalblue',  # Even lighter blue
    4: 'cornflowerblue',  # Lighter blue
    5: 'slateblue',  # Lightest blue
    6: 'red',  # Red
    7: 'tomato',  # Lighter red
    8: 'coral'   # Lightest red
}


for cluster in np.arange(1,9):
    fig, ax = plt.subplots()
    fig.set_size_inches(10,10)
    fig.suptitle(f'cluster {cluster}')#, units {len(unit_ids)}, sessions {num_sessions}')

    for experience, color in zip(['Familiar', 'Novel'], ['b', 'r']):
        unit_ids = vbn_utils.get_unit_ids(units, ['SCMRN',], clusters=cluster, clustering='new', experience=experience)
        # unit_filter = du.get_units_in_cluster(units, cluster, clustering='new') & \
        #         du.apply_unit_quality_filter(units) & \
        #         du.getUnitsInRegion(units, 'SCMRN')
        # unit_ids = units.loc[unit_filter]['unit_id'].values #[1076896692, 1068232021, 1068231696,]#
        if len(unit_ids)==0:
            continue
        num_sessions = units.set_index('unit_id').loc[unit_ids]['ecephys_session_id'].unique().size

        stim_filter_conds = {'shared_hit': ['engaged', 'is_change', '~omitted', '~previous_omitted', 'hit', 'is_shared'],
                            'nonshared_hit': ['engaged', 'is_change', '~omitted', '~previous_omitted', 'hit', '~is_shared'],
                            'shared_nonchange_lick': ['engaged', '~is_change', '~omitted', '~previous_omitted', 'is_shared', 'flashes_since_change>5', 'flashes_since_last_lick>1', 'lickbout_for_flash_during_response_window'],
                            'nonshared_nonchange_lick': ['engaged', '~is_change', '~omitted', '~previous_omitted', '~is_shared', 'flashes_since_change>5', 'flashes_since_last_lick>1', 'lickbout_for_flash_during_response_window'],
                            'shared_nonchange_nolick': ['engaged', '~is_change', '~omitted', '~previous_omitted', 'is_shared', 'flashes_since_change>5', 'flashes_since_last_lick>1', '~lickbout_for_flash_during_response_window'],
                            'nonshared_nonchange_nolick': ['engaged', '~is_change', '~omitted', '~previous_omitted', '~is_shared', 'flashes_since_change>5', 'flashes_since_last_lick>1', '~lickbout_for_flash_during_response_window'],
                            }
        #stim_filter_cond2 = ['engaged', '~is_change', '~omitted', '~previous_omitted', 'flashes_since_change>5', 'flashes_since_last_lick>1', 'lickbout_for_flash_during_response_window', "image_name=='im036_r'"]

        cluster_means = []
        for label, stim_filter in stim_filter_conds.items():
            if label=='shared_hit':
                alpha = 1
                ls = 'dotted'
            
            elif label=='nonshared_hit':
                alpha = 1
                ls = 'solid'
            
            elif label=='shared_nonchange_lick': 
                alpha = 0.5
                ls = 'dotted'

            elif label=='nonshared_nonchange_lick': 
                alpha = 0.5
                ls = 'solid'
            
            elif label=='nonshared_nonchange_nolick': 
                alpha = 0.25
                ls = 'solid'
            
            elif label=='shared_nonchange_nolick': 
                alpha = 0.25
                ls = 'dotted'


            unit_data, unitIds = vbn_utils.unit_averaged_psth(active_tensor_file, 
                                        stim_table, 
                                        session_list, 
                                        unit_ids, 
                                        *stim_filter, 
                                        baseline_length=750, 
                                        resp_window_length=750)
            concat = np.concatenate(unit_data)
            concat = np.array([exponential_convolve(c, 5, symmetrical=True) for c in concat])
            if concat.size==0:
                continue
            
            if base_sub:
                concat = concat - concat[:, :50].mean(axis=1)[:, None]
            # concat = np.array([np.nanmean(u, axis=0) for u in unit_data])
            # flash_data[cluster].append(concat)
            #vbn_utils.mean_sem_plot(concat*1000, ax, time,)# color=colors[istim])
            ax.plot(time, np.nanmean(concat, axis=0), label=label, color=color, alpha=alpha, ls=ls,)
            ax.set_xlabel('Time from flash (ms)')
            cluster_means.append(np.nanmean(concat[:, 650:750], axis=1))
        
        # cluster_means = np.array(cluster_means)
        # cluster_means = np.roll(cluster_means, shift=-1, axis=0)
        # cluster_means = cluster_means - cluster_means[0][None,:]
        # cluster_mean = np.nanmean(cluster_means, axis=1)
        # cluster_sem = np.nanstd(cluster_means, axis=1)/cluster_means.shape[1]**0.5
        # ax_agg.plot(np.arange(2,12), cluster_mean, 'o', color=cluster_colors[cluster], label=cluster)
        # ax_agg.errorbar(np.arange(2,12), cluster_mean, yerr=cluster_sem, color=cluster_colors[cluster])
        # ax_agg.set_xticks(np.arange(2,12))
        # ax_agg.set_xticklabels(list(np.arange(2,10)) + ['false alarm', 'hit'])
        # ax.set_xlim(-400, 350)
        # ax.legend()
        # ax_agg.legend()

In [ ]:
import warnings
warnings.filterwarnings("ignore")

fig_agg, ax_agg = plt.subplots()

base_sub = False
session_list = list(active_tensor.keys())
time = np.arange(-750, 750)

cluster_colors = {
    1: 'navy',  # Blue
    2: 'mediumblue',  # Lighter blue
    3: 'royalblue',  # Even lighter blue
    4: 'cornflowerblue',  # Lighter blue
    5: 'slateblue',  # Lightest blue
    6: 'red',  # Red
    7: 'tomato',  # Lighter red
    8: 'coral'   # Lightest red
}


for cluster in np.arange(1,9):
# flash_data = {cluster:[] for cluster in np.arange(15)}
    unit_filter = du.get_units_in_cluster(units, cluster, clustering='new') & du.apply_unit_quality_filter(units) & \
                    du.getUnitsInRegion(units, 'VISall')
    unit_ids = units.loc[unit_filter]['unit_id'].values #[1076896692, 1068232021, 1068231696,]#
    if len(unit_ids)==0:
        continue
    num_sessions = units.loc[unit_filter]['ecephys_session_id'].unique().size


    fig, ax = plt.subplots()
    fig.set_size_inches(10,10)
    fig.suptitle(f'cluster {cluster}, units {len(unit_ids)}, sessions {num_sessions}')
    stim_filter_cond = ['engaged', '~is_change', '~omitted', '~previous_omitted', 'flashes_since_change>5']
    #stim_filter_cond2 = ['engaged', '~is_change', '~omitted', '~previous_omitted', 'flashes_since_change>5', 'flashes_since_last_lick>1', 'lickbout_for_flash_during_response_window', "image_name=='im036_r'"]

    cluster_means = []
    for flashes_since_lick in range(11):
        if flashes_since_lick==0:
            stim_filter = ['engaged', 'is_change', 'hit']
            label = 'change lick'
            alpha = 1
            color = 'g'
            ls = 'solid'
        elif flashes_since_lick==1:
            continue
        elif flashes_since_lick<10:
            filter_add_on = [f'flashes_since_last_lick=={flashes_since_lick}', '~lickbout_for_flash_during_response_window']
            stim_filter =  stim_filter_cond + filter_add_on
            label = f'{flashes_since_lick}'
            alpha = flashes_since_lick/9
            color = 'k'
            ls = 'solid'
        elif flashes_since_lick==10: 
            filter_add_on = ['lickbout_for_flash_during_response_window']
            stim_filter =  stim_filter_cond + filter_add_on
            label = 'non change lick'
            color = 'g'
            alpha = 1
            ls = 'dotted'

        unit_data, unitIds = vbn_utils.unit_averaged_psth(active_tensor_file, 
                                    stim_table, 
                                    session_list, 
                                    unit_ids, 
                                    *stim_filter, 
                                    baseline_length=750, 
                                    resp_window_length=750)
        concat = np.concatenate(unit_data)
        concat = np.array([exponential_convolve(c, 5, symmetrical=True) for c in concat])
        if concat.size==0:
            continue
        
        if base_sub:
            concat = concat - concat[:, :50].mean(axis=1)[:, None]
        # concat = np.array([np.nanmean(u, axis=0) for u in unit_data])
        # flash_data[cluster].append(concat)
        #vbn_utils.mean_sem_plot(concat*1000, ax, time,)# color=colors[istim])
        ax.plot(time, np.nanmean(concat, axis=0), label=label, color=color, alpha=alpha, ls=ls,)
        ax.set_xlabel('Time from flash (ms)')
        cluster_means.append(np.nanmean(concat[:, 650:750], axis=1))
    
    cluster_means = np.array(cluster_means)
    cluster_means = np.roll(cluster_means, shift=-1, axis=0)
    cluster_means = cluster_means - cluster_means[0][None,:]
    cluster_mean = np.nanmean(cluster_means, axis=1)
    cluster_sem = np.nanstd(cluster_means, axis=1)/cluster_means.shape[1]**0.5
    ax_agg.plot(np.arange(2,12), cluster_mean, 'o', color=cluster_colors[cluster], label=cluster)
    ax_agg.errorbar(np.arange(2,12), cluster_mean, yerr=cluster_sem, color=cluster_colors[cluster])
    ax_agg.set_xticks(np.arange(2,12))
    ax_agg.set_xticklabels(list(np.arange(2,10)) + ['false alarm', 'hit'])
    ax.set_xlim(-400, 350)
    ax.legend()
    ax_agg.legend()

In [ ]:
cluster_means = np.array(cluster_means)
cluster_means = cluster_means - cluster_means[2][None,:]
cluster_mean = np.nanmean(cluster_means, axis=1)
cluster_sem = np.nanstd(cluster_means, axis=1)/cluster_means.shape[1]**0.5
plt.plot(np.arange(3), cluster_mean, 'o', color=cluster_colors[cluster], label=cluster)
plt.errorbar(np.arange(3), cluster_mean, yerr=cluster_sem, color=cluster_colors[cluster])

### Split by behavioral strategy (timing vs. visual)

In [ ]:
timing_sessions = sessions_table[sessions_table['strategy']=='timing']['ecephys_session_id'].values
visual_sessions = sessions_table[sessions_table['strategy']=='visual']['ecephys_session_id'].values

In [ ]:
import warnings
warnings.filterwarnings("ignore")
base_sub = False

for session_subset in [timing_sessions, visual_sessions]:
    session_list = list(active_tensor.keys())
    session_list = [s for s in session_list if int(s) in timing_sessions]
    time = np.arange(-50, 750)
    for cluster in np.arange(1, 12):
    # flash_data = {cluster:[] for cluster in np.arange(15)}
        unit_filter = du.get_units_in_cluster(units, cluster, clustering='new') & du.apply_unit_quality_filter(units) & \
                                        (units['ecephys_session_id'].isin(session_list))
        unit_ids = units.loc[unit_filter]['unit_id'].values #[1076896692, 1068232021, 1068231696,]#
        num_sessions = units.loc[unit_filter]['ecephys_session_id'].unique().size


        fig, ax = plt.subplots()
        fig.suptitle(f'cluster {cluster}, units {len(unit_ids)}, sessions {num_sessions}')
        stim_filter_cond = ['engaged', '~is_change', '~omitted', '~previous_omitted', 'flashes_since_change>5']
        #stim_filter_cond2 = ['engaged', '~is_change', '~omitted', '~previous_omitted', 'flashes_since_change>5', 'flashes_since_last_lick>1', 'lickbout_for_flash_during_response_window', "image_name=='im036_r'"]

        colors = ['k', 'gray']
        for flashes_since_lick in range(11):
            
            filter_add_on = [f'flashes_since_last_lick=={flashes_since_lick}', '~lickbout_for_flash_during_response_window'] if flashes_since_lick<10 else ['lickbout_for_flash_during_response_window']
            label = f'{flashes_since_lick}' if flashes_since_lick<10 else 'lick'
            stim_filter =  stim_filter_cond + filter_add_on
            unit_data, unitIds = vbn_utils.unit_averaged_psth(active_tensor_file, 
                                        stim_table, 
                                        session_list, 
                                        unit_ids, 
                                        *stim_filter, 
                                        baseline_length=50, 
                                        resp_window_length=750)
            concat = np.concatenate(unit_data)
            concat = np.array([exponential_convolve(c, 3, symmetrical=True) for c in concat])
            if base_sub:
                concat = concat - concat[:, :50].mean(axis=1)[:, None]
            # concat = np.array([np.nanmean(u, axis=0) for u in unit_data])
            # flash_data[cluster].append(concat)
            #vbn_utils.mean_sem_plot(concat*1000, ax, time,)# color=colors[istim])
            ax.plot(time, np.nanmean(concat, axis=0), label=label)
            ax.set_xlabel('Time from flash (ms)')

        ax.legend()

In [ ]:
import warnings
warnings.filterwarnings("ignore")
base_sub = True
session_list = list(active_tensor.keys())
time = np.arange(-50, 750)
for cluster in [5,7,]:#np.arange(1, 12):
# flash_data = {cluster:[] for cluster in np.arange(15)}
    unit_filter = du.get_units_in_cluster(units, cluster, clustering='new') & \
                        du.apply_unit_quality_filter(units) & du.getUnitsInRegion(units, 'VISall')
    unit_ids = units.loc[unit_filter]['unit_id'].values #[1076896692, 1068232021, 1068231696,]#
    num_sessions = units.loc[unit_filter]['ecephys_session_id'].unique().size


    fig, ax = plt.subplots()
    fig.suptitle(f'cluster {cluster}, units {len(unit_ids)}, sessions {num_sessions}')
    stim_filter_cond = ['engaged', '~is_change', '~omitted', '~previous_omitted', 'flashes_since_change>5']
    #stim_filter_cond2 = ['engaged', '~is_change', '~omitted', '~previous_omitted', 'flashes_since_change>5', 'flashes_since_last_lick>1', 'lickbout_for_flash_during_response_window', "image_name=='im036_r'"]

    colors = ['k', 'gray']
    for flashes_since_lick in range(11):
        
        filter_add_on = [f'flashes_since_last_lick=={flashes_since_lick}', '~lickbout_for_flash_during_response_window'] if flashes_since_lick<10 else ['lickbout_for_flash_during_response_window']
        label = f'{flashes_since_lick}' if flashes_since_lick<10 else 'lick'
        stim_filter =  stim_filter_cond + filter_add_on
        unit_data, unitIds = vbn_utils.unit_averaged_psth(active_tensor_file, 
                                    stim_table, 
                                    session_list, 
                                    unit_ids, 
                                    *stim_filter, 
                                    baseline_length=50, 
                                    resp_window_length=750)
        concat = np.concatenate(unit_data)
        concat = np.array([exponential_convolve(c, 3, symmetrical=True) for c in concat])
        if concat.size==0:
            continue
        
        if base_sub:
            concat = concat - concat[:, :50].mean(axis=1)[:, None]
        # concat = np.array([np.nanmean(u, axis=0) for u in unit_data])
        # flash_data[cluster].append(concat)
        #vbn_utils.mean_sem_plot(concat*1000, ax, time,)# color=colors[istim])
        ax.plot(time, np.nanmean(concat, axis=0), label=label)
        ax.set_xlabel('Time from flash (ms)')

    ax.legend()

In [ ]:
import warnings
warnings.filterwarnings("ignore")
base_sub = False
time = np.arange(-750, 750)
session_labels = ['timing', 'visual']
all_session_list = list(active_tensor.keys())
for cluster in np.arange(1,13):
    fig, axes = plt.subplots(1,2)
    fig.set_size_inches(12,5)
    for isess, session_subset in enumerate([timing_sessions, visual_sessions]):
        session_list = [s for s in all_session_list if int(s) in session_subset]
        
        unit_filter = du.get_units_in_cluster(units, cluster, clustering='new') & du.apply_unit_quality_filter(units) & \
                                        (units['ecephys_session_id'].isin([int(s) for s in session_list])) & \
                                        du.getUnitsInRegion(units, 'VISall')
        unit_ids = units.loc[unit_filter]['unit_id'].values #[1076896692, 1068232021, 1068231696,]#
        num_sessions = units.loc[unit_filter]['ecephys_session_id'].unique().size


        # stim_filter_cond = ['engaged', 'is_prechange']
        # stim_filter_cond2 = ['engaged', 'is_change']
        stim_filter_cond = ['engaged', '~is_change', '~omitted', '~previous_omitted', 'flashes_since_change>5', 'lickbout_for_flash_during_response_window',]
        stim_filter_cond2 = ['engaged', '~is_change', '~omitted', '~previous_omitted', 'flashes_since_change>5', 'flashes_since_last_lick>1', '~lickbout_for_flash_during_response_window',]

        colors = ['k', 'gray']
        for isc, stim_filter in enumerate([stim_filter_cond, stim_filter_cond2]):
            
            label = isc #stim_filter
            unit_data, unitIds = vbn_utils.unit_averaged_psth(active_tensor_file, 
                                        stim_table, 
                                        session_list, 
                                        unit_ids, 
                                        *stim_filter, 
                                        baseline_length=750, 
                                        resp_window_length=750)
            concat = np.concatenate(unit_data)
            concat = np.array([exponential_convolve(c, 3, symmetrical=True) for c in concat])
            if base_sub:
                concat = concat - concat[:, :50].mean(axis=1)[:, None]
            
            if concat.size ==0:
                continue
            # concat = np.array([np.nanmean(u, axis=0) for u in unit_data])
            # flash_data[cluster].append(concat)
            #vbn_utils.mean_sem_plot(concat*1000, ax, time,)# color=colors[istim])
            axes[isess].plot(time, np.nanmean(concat, axis=0), color=colors[isc], label=label)
            axes[isess].set_xlabel('Time from flash (ms)')

        axes[isess].legend()
        axes[isess].set_title(f'{cluster} {session_labels[isess]} n: {len(unit_ids)}')

In [ ]:
import warnings
warnings.filterwarnings("ignore")
base_sub = False
session_list = list(active_tensor.keys())
plot_window = slice(0, 250)
time = np.arange(-50, 750)[plot_window]

clusters_to_plot = [7]
for cluster in clusters_to_plot:

    unit_filter = du.get_units_in_cluster(units, cluster, clustering='new') & \
                        du.apply_unit_quality_filter(units)# & du.getUnitsInRegion(units, 'VISall')
    unit_ids = units.loc[unit_filter]['unit_id'].values #[1076896692, 1068232021, 1068231696,]#
    num_sessions = units.loc[unit_filter]['ecephys_session_id'].unique().size

    fig, ax = plt.subplots(1,3)
    fig.suptitle(f'cluster {cluster}, units {len(unit_ids)}, sessions {num_sessions}')

    
    # Active/Passive comparison
    stim_filter = ['engaged', '~is_change', '~omitted', '~previous_omitted', 'flashes_since_change>5']
    for tensor_file, color in zip([active_tensor_file, passive_tensor_file], ['k', 'gray']):
        unit_data, unitIds = vbn_utils.unit_averaged_psth(tensor_file, 
                                    stim_table, 
                                    session_list, 
                                    unit_ids, 
                                    *stim_filter, 
                                    baseline_length=50, 
                                    resp_window_length=750)
        concat = np.concatenate(unit_data)
        concat = np.array([exponential_convolve(c, 3, symmetrical=True) for c in concat])
        if concat.size==0:
            continue
        
        if base_sub:
            concat = concat - concat[:, :50].mean(axis=1)[:, None]

        vbn_utils.mean_sem_plot(concat[:, plot_window]*1000, ax[0], time, color=color)
    
    # Lick/No-lick comparison
    stim_filters = (
                ['engaged', '~is_change', '~omitted', '~previous_omitted', 'flashes_since_change>5', 'lickbout_for_flash_during_response_window'],
                ['engaged', '~is_change', '~omitted', '~previous_omitted', 'flashes_since_change>5', '~lickbout_for_flash_during_response_window'],
                    )
    for stim_filter, color in zip(stim_filters, ['k', 'gray']):
        unit_data, unitIds = vbn_utils.unit_averaged_psth(active_tensor_file, 
                                    stim_table, 
                                    session_list, 
                                    unit_ids, 
                                    *stim_filter, 
                                    baseline_length=50, 
                                    resp_window_length=750)
        concat = np.concatenate(unit_data)
        concat = np.array([exponential_convolve(c, 3, symmetrical=True) for c in concat])
        if concat.size==0:
            continue
        
        if base_sub:
            concat = concat - concat[:, :50].mean(axis=1)[:, None]

        vbn_utils.mean_sem_plot(concat[:, plot_window]*1000, ax[1], time, color=color)


    # stim_filter_cond = ['engaged', '~is_change', '~omitted', '~previous_omitted', 'flashes_since_change>5']
    # #stim_filter_cond2 = ['engaged', '~is_change', '~omitted', '~previous_omitted', 'flashes_since_change>5', 'flashes_since_last_lick>1', 'lickbout_for_flash_during_response_window', "image_name=='im036_r'"]

    # colors = ['k', 'gray']
    # for flashes_since_lick in range(11):
        
    #     filter_add_on = [f'flashes_since_last_lick=={flashes_since_lick}', '~lickbout_for_flash_during_response_window'] if flashes_since_lick<10 else ['lickbout_for_flash_during_response_window']
    #     label = f'{flashes_since_lick}' if flashes_since_lick<10 else 'lick'
    #     stim_filter =  stim_filter_cond + filter_add_on
    #     unit_data, unitIds = vbn_utils.unit_averaged_psth(active_tensor_file, 
    #                                 stim_table, 
    #                                 session_list, 
    #                                 unit_ids, 
    #                                 *stim_filter, 
    #                                 baseline_length=50, 
    #                                 resp_window_length=750)
    #     concat = np.concatenate(unit_data)
    #     concat = np.array([exponential_convolve(c, 3, symmetrical=True) for c in concat])
    #     if concat.size==0:
    #         continue
        
    #     if base_sub:
    #         concat = concat - concat[:, :50].mean(axis=1)[:, None]
    #     # concat = np.array([np.nanmean(u, axis=0) for u in unit_data])
    #     # flash_data[cluster].append(concat)
    #     #vbn_utils.mean_sem_plot(concat*1000, ax, time,)# color=colors[istim])
    #     ax.plot(time, np.nanmean(concat, axis=0), label=label)
    #     ax.set_xlabel('Time from flash (ms)')

    # ax.legend()

## Response latencies

In [ ]:
import warnings
warnings.filterwarnings("ignore")
base_sub = False
session_list = list(active_tensor.keys())
plot_window = slice(50, 150)
time = np.arange(-50, 750)[plot_window]

cluster_colors = {1:'k', 2:'g', 6:'r'}
alphas = {'LGd': 1,
        'VISall': 0.25,
        'VISam': 0.5,
        'VISp': 0.75,
        'SCMRN': 0.1
    }
clusters_to_plot = [1,2,6]
areas_to_plot = ['LGd','VISp', 'VISam', 'SCMRN']

fig, ax = plt.subplots(1,2)
fig.set_size_inches(10,5)
# fig.suptitle(f'cluster {cluster}, units {len(unit_ids)}, sessions {num_sessions}')
for cluster in clusters_to_plot:
    for area in areas_to_plot:
        if cluster==6 and area != 'SCMRN':
            continue
        color = cluster_colors[cluster]
        alpha = alphas[area]
        label = f'{area}_{cluster}'
        unit_filter = du.get_units_in_cluster(units, cluster, clustering='new') & \
                            du.apply_unit_quality_filter(units) & du.getUnitsInRegion(units, area)
        unit_ids = units.loc[unit_filter]['unit_id'].values #[1076896692, 1068232021, 1068231696,]#
        num_sessions = units.loc[unit_filter]['ecephys_session_id'].unique().size

        
        # Active/Passive comparison
        stim_filter = ['engaged', '~is_change', '~omitted', '~previous_omitted', 
            'flashes_since_change>5', '~lickbout_for_flash_during_response_window']
        # for tensor_file, color in zip([active_tensor_file, passive_tensor_file], ['k', 'gray']):
        unit_data, unitIds = vbn_utils.unit_averaged_psth(active_tensor_file, 
                                    stim_table, 
                                    session_list, 
                                    unit_ids, 
                                    *stim_filter, 
                                    baseline_length=50, 
                                    resp_window_length=750)
        concat = np.concatenate(unit_data)
        concat = np.array([exponential_convolve(c, 3, symmetrical=True) for c in concat])
        if concat.size==0:
            continue
        
        vbn_utils.mean_sem_plot(concat[:, plot_window]*1000, ax[0], time, color=color, alpha=alpha, label=label)

        concat = concat - concat[:, :50].mean(axis=1)[:, None]
        # concat = concat/concat.max(axis=1)[:, None]
        vbn_utils.mean_sem_plot(concat[:, plot_window], ax[1], time, color=color, alpha=alpha)

ax[0].legend()

In [ ]:
import warnings
warnings.filterwarnings("ignore")
base_sub = False
session_list = list(active_tensor.keys())
plot_window = slice(50, 150)
time = np.arange(-50, 750)[plot_window]

cluster_colors = {1:'k', 2:'g', 6:'r'}
alphas = {'LGd': 1,
        'VISall': 0.25,
        'VISam': 0.5,
        'VISp': 0.75,
        'SCMRN': 0.1
    }
clusters_to_plot = [1,2,6]
areas_to_plot = ['LGd', 'VISp', 'VISl', 'VISrl', 'VISal', 'LP', 'VISpm', 'VISam', 'SCMRN']

medians = {}
for cluster in clusters_to_plot:
    for area in areas_to_plot:
        if cluster==6 and area != 'SCMRN':
            continue

        fig, ax = plt.subplots()
        fig.set_size_inches(10,5)
        fig.suptitle(f'{area} {cluster}')

    
        unit_filter = du.get_units_in_cluster(units, cluster, clustering='new') & \
                            du.apply_unit_quality_filter(units) & du.getUnitsInRegion(units, area)
        unit_ids = units.loc[unit_filter]['unit_id'].values #[1076896692, 1068232021, 1068231696,]#
        num_sessions = units.loc[unit_filter]['ecephys_session_id'].unique().size

        
        # Active/Passive comparison
        stim_filter = ['engaged', '~is_change', '~omitted', '~previous_omitted', 
            'flashes_since_change>5', '~lickbout_for_flash_during_response_window']
        # for tensor_file, color in zip([active_tensor_file, passive_tensor_file], ['k', 'gray']):
        unit_data, unitIds = vbn_utils.unit_averaged_psth(active_tensor_file, 
                                    stim_table, 
                                    session_list, 
                                    unit_ids, 
                                    *stim_filter, 
                                    baseline_length=50, 
                                    resp_window_length=750,
                                    time_to_first_spike=True)
        
        across_sessions = [val for sub in unit_data for val in sub]
        ax.hist(across_sessions, bins = np.arange(0,500,10))

        medians[f'{area}_{cluster}'] = np.nanmedian(across_sessions)
# ax[0].legend()

fig, ax = plt.subplots()
for ind, (ac, val) in enumerate(medians.items()):
    ax.plot(ind, val, 'ko')

ax.set_xticks(np.arange(len(medians.keys())))
ax.set_xticklabels(medians.keys(), rotation=90)


In [ ]:
import warnings
warnings.filterwarnings("ignore")
base_sub = False
session_list = list(active_tensor.keys())
plot_window = slice(50, 150)
time = np.arange(-50, 750)[plot_window]

cell_type_colors = {'RS':'k', 'FS':'r', 'VIP':'purple', 'SST':'green'}
# alphas = {'LGd': 1,
#         'VISall': 0.25,
#         'VISam': 0.5,
#         'VISp': 0.75,
#         'SCMRN': 0.1
#     }
cell_types_to_plot = ['RS', 'FS', 'SST', 'VIP']
areas_to_plot = ['VISall', ]#'VISp']

fig, ax = plt.subplots(1,2)
fig.set_size_inches(10,5)
# fig.suptitle(f'cluster {cluster}, units {len(unit_ids)}, sessions {num_sessions}')
for cell_type in cell_types_to_plot:
    for area in areas_to_plot:

        color = cell_type_colors[cell_type]
        alpha = 1 #alphas[area]
        label = f'{area}_{cell_type}'
        # unit_filter = du.apply_unit_quality_filter(units) & du.getUnitsInRegion(units, area, cell_type=cell_type)
        # unit_ids = units.loc[unit_filter]['unit_id'].values #[1076896692, 1068232021, 1068231696,]#
        unit_ids = vbn_utils.get_unit_ids(units, area, cell_type,)# experience='Novel')
        num_sessions = units.loc[unit_filter]['ecephys_session_id'].unique().size

        
        # Active/Passive comparison
        stim_filter = ['engaged', '~is_change', '~omitted', '~previous_omitted', 
            'flashes_since_change>5', '~lickbout_for_flash_during_response_window', '~is_shared']
        stim_filter = ['engaged', 'is_change', 'hit', '~is_shared']
        # for tensor_file, color in zip([active_tensor_file, passive_tensor_file], ['k', 'gray']):
        unit_data, unitIds = vbn_utils.unit_averaged_psth(active_tensor_file, 
                                    stim_table, 
                                    session_list, 
                                    unit_ids, 
                                    *stim_filter, 
                                    baseline_length=50, 
                                    resp_window_length=750)
        concat = np.concatenate(unit_data)
        concat = np.array([exponential_convolve(c, 3, symmetrical=True) for c in concat])
        if concat.size==0:
            continue
        
        vbn_utils.mean_sem_plot(concat[:, plot_window]*1000, ax[0], time, color=color, alpha=alpha, label=label)

        concat = concat - concat[:, :50].mean(axis=1)[:, None]
        # concat = concat/concat.max(axis=1)[:, None]
        vbn_utils.mean_sem_plot(concat[:, plot_window], ax[1], time, color=color, alpha=alpha)

ax[0].legend()

In [ ]:
import warnings
warnings.filterwarnings("ignore")
base_sub = False
session_list = list(active_tensor.keys())
plot_window = slice(50, 150)
time = np.arange(-50, 750)[plot_window]

cell_type_colors = {'RS':'k', 'FS':'r', 'VIP':'purple', 'SST':'green'}
# alphas = {'LGd': 1,
#         'VISall': 0.25,
#         'VISam': 0.5,
#         'VISp': 0.75,
#         'SCMRN': 0.1
#     }
cell_types_to_plot = ['RS', 'FS', 'SST', 'VIP']
areas_to_plot = ['VISall', 'VISp']

fig, ax = plt.subplots()
fig.set_size_inches(10,5)
# fig.suptitle(f'cluster {cluster}, units {len(unit_ids)}, sessions {num_sessions}')
medians = {}
for cell_type in cell_types_to_plot:
    for area in areas_to_plot:

        color = cell_type_colors[cell_type]
        alpha = 1 #alphas[area]
        label = f'{area}_{cell_type}'
        # unit_filter = du.apply_unit_quality_filter(units) & du.getUnitsInRegion(units, area, cell_type=cell_type)
        # unit_ids = units.loc[unit_filter]['unit_id'].values #[1076896692, 1068232021, 1068231696,]#
        unit_ids = vbn_utils.get_unit_ids(units, area, cell_type, experience='Novel')
        num_sessions = units.loc[unit_filter]['ecephys_session_id'].unique().size

        
        # Active/Passive comparison
        stim_filter = ['engaged', '~is_change', '~omitted', '~previous_omitted', 
            'flashes_since_change>5', '~lickbout_for_flash_during_response_window', '~is_shared']
        # for tensor_file, color in zip([active_tensor_file, passive_tensor_file], ['k', 'gray']):
        unit_data, unitIds = vbn_utils.unit_averaged_psth(active_tensor_file, 
                                    stim_table, 
                                    session_list, 
                                    unit_ids, 
                                    *stim_filter, 
                                    baseline_length=50, 
                                    resp_window_length=750,
                                    time_to_first_spike=True)
        
        across_sessions = [val for sub in unit_data for val in sub]
        # ax.hist(across_sessions, bins = np.arange(0,500,10))
        cumulative_hist = vbn_utils.cumulative_hist(across_sessions)
        ax.plot(cumulative_hist[0], cumulative_hist[1], color=cell_type_colors[cell_type])

        medians[f'{area}_{cell_type}'] = np.nanmedian(across_sessions)
# ax[0].legend()

fig, ax = plt.subplots()
for ind, (ac, val) in enumerate(medians.items()):
    ax.plot(ind, val, 'ko')

ax.set_xticks(np.arange(len(medians.keys())))
ax.set_xticklabels(medians.keys(), rotation=90)


## Lick-aligned unit activity

### Compute lick-aligned responses (if pre-computed, load below)

In [ ]:
unit_filter = du.apply_unit_quality_filter(units) #& du.getUnitsInRegion(units, 'VISall')
unit_ids = units.loc[unit_filter]['unit_id'].values 
session_list = list(active_tensor.keys())

lick_aligned_unit_summary = {u:[] for u in unit_ids}
stim_filter = ['engaged', '~is_change', '~omitted', '~previous_omitted', 'flashes_since_change>5', 'lickbout_for_flash_during_response_window']
unit_data, shuffle_data, unitIds = vbn_utils.unit_averaged_psth_lick_aligned(active_tensor_file, 
                                    stim_table, 
                                    session_list, 
                                    unit_ids, 
                                    *stim_filter, 
                                    baseline_length=750, 
                                    resp_window_length=1500)
    

In [ ]:
for udata, sdata, uids in zip(unit_data, shuffle_data, unitIds):
    if len(uids)>0:
        for ud, sd, uid in zip(udata, sdata, uids):
            umean = exponential_convolve(ud, 3, symmetrical=True)
            time_above = np.convolve(umean>sd[2], np.ones(10)).max()
            time_below = np.convolve(umean<sd[1], np.ones(10)).max()
            passes = np.max((time_above, time_below))>9
            lick_aligned_unit_summary[uid] = {'mean': umean, 
                                              'shuffle_mean': sd[0],
                                              'shuffle_ci_low': sd[1],
                                              'shuffle_ci_high': sd[2],
                                              'pass': passes}

### Load pre-computed lick-aligned data

In [ ]:
import pickle
with open("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/lick_aligned_unit_data.pkl", 'rb') as file:
    lick_aligned_unit_summary = pickle.load(file)

In [ ]:
lick_aligned_df = pd.DataFrame.from_dict(lick_aligned_unit_summary, orient='index')

In [ ]:
lick_aligned_df = lick_aligned_df.merge(units[['unit_id', 'cluster_labels_new', 'structure_acronym']], left_index=True, right_on='unit_id')

In [ ]:
lick_aligned_df.pivot_table(index='cluster_labels_new', values = 'high_pass')

In [ ]:
lick_aligned_df.replace(['SCig', 'SCiw'], 'SCm', inplace=True)

In [ ]:
area_pt = lick_aligned_df.pivot_table(index='structure_acronym', values = 'high_pass', aggfunc=['mean', 'count'])

In [ ]:
area_pt[area_pt[('count', 'high_pass')]>1000].sort_values(by=('mean', 'high_pass'))

In [ ]:
c1 = lick_aligned_df[lick_aligned_df['high_pass'] & (lick_aligned_df['cluster_labels_new']==8)]

In [ ]:
lick_aligned_df['high_conv'] = lick_aligned_df.apply(lambda row: np.convolve(row['mean']>row['shuffle_ci_high'], np.ones(10)).max(), axis=1)

In [ ]:
lick_aligned_df['high_pass'] = lick_aligned_df['high_conv']>9

In [ ]:
shuff = np.concatenate(shuffle_data, axis=0).squeeze()
shuff = np.array([exponential_convolve(c, 3, symmetrical=True) for c in shuff])

In [ ]:
lows = np.percentile(shuff, 0.05, axis=0)
highs = np.percentile(shuff, 99.95, axis=0)

In [ ]:
concat = np.concatenate(unit_data)
concat = np.array([exponential_convolve(c, 3, symmetrical=True) for c in concat])
plt.plot(np.mean(concat,axis=0))
# plt.plot(np.mean(shuff, axis=0))
# plt.plot(lows, 'k', ls='dotted')
# plt.plot(highs, 'k', ls='dotted')
# plt.plot(exponential_convolve(low, 3, symmetrical=True), 'r', ls='dotted')
# plt.plot(exponential_convolve(high, 3, symmetrical=True), 'r', ls='dotted')
shuffle_concat = np.concatenate(shuffle_data)
plt.plot(shuffle_concat[0,0])
plt.plot(shuffle_concat[0,1])
plt.plot(shuffle_concat[0,2])



## LGd response latency

In [ ]:
lgd_units = vbn_utils.get_unit_ids(units, ['LGd',], clusters=[1], clustering='new')
session_id = '1093642839'

stims = stim_table[stim_table['session_id']==int(session_id)].reset_index()
stim_filter = ['engaged', 'is_change',]# '~omitted', '~previous_omitted', 'flashes_since_change>5', '~lickbout_for_flash_during_response_window']
chained_query = ' & '.join(stim_filter)
stims_subset = stims.query(chained_query)
flash_indices = stims_subset.index.values

ttfs, unitIDs = vbn_utils.get_time_to_first_spike(session_id, active_tensor_file, lgd_units, flash_indices, baseline_length = 50, resp_window_length=750)


In [ ]:
ttfs=np.array(ttfs)
np.median(ttfs[ttfs<200])

In [ ]:
plt.hist(ttfs, bins=np.arange(0, 500, 10))